<a href="https://colab.research.google.com/github/vituhaa/Multimodal-Reasoning-for-STEM/blob/main/Multimodal_Reasoning_for_STEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Qwen/Qwen3-VL-2B-Instruct

Qwen/Qwen3-VL-4B-Instruct

HuggingFaceTB/SmolVLM-256M Instruct

google/t5gemma-2-270m-270m

"HuggingFaceTB/SmolVLM-256M Instruct" model was selected to fine-tune.

Zero-shot inference

In [1]:
! nvidia-smi

Fri Mar 13 11:26:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
! pip install -q torch torchvision
! pip install -q transformers accelerate
! pip install --upgrade pip wheel setuptools
! pip install pillow
! MAX_JOBS=64 pip install "flash-attn<2.0.0" --no-build-isolation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.5 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 69.0 MB/s  0:00:00
  Preparing metadata (pyproject.toml) ... canceled
ERROR: Operation cancelled by user
^C


In [5]:
import torch
import requests
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers.image_utils import load_image
from io import BytesIO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

image_path = "/content/formula.jpg"
image = load_image(image_path)

model_name = "HuggingFaceTB/SmolVLM-256M-Instruct"
processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-256M-Instruct")
model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    _attn_implementation="sdpa"
).to(DEVICE)

input_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this hand-written mathematical formula into LaTex format."}
        ]
    }
]

prompt = processor.apply_chat_template(input_messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

generated_ids = model.generate(**inputs, max_new_tokens=500)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

print(generated_texts[0])

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

User:

Convert this hand-written mathematical formula into LaTex format.
Assistant: sinx-siny-sin(x-y)
